In [ ]:
!pip install pretty_midi

In [1]:
import pretty_midi as pm

In [ ]:
# load midi file to train LSTM model
midi_file = "sample.mid"
midi_data = pm.PrettyMIDI(midi_file)

In [3]:
# extract notes
from pretty_midi.utilities import note_number_to_name

note_sequence = []
note_sequence_numeric = []
for instrument in midi_data.instruments:
  if not instrument.is_drum:
    for note in instrument.notes:
      note_name = note_number_to_name(note.pitch)
      note_sequence.append(note_name)
      note_sequence_numeric.append(note.pitch)

print(note_sequence)
print(note_sequence_numeric)

['D4', 'E4', 'F#4', 'C#4', 'C#5', 'F#5', 'A5', 'C#6', 'A5', 'D5', 'F5', 'G#5', 'A5', 'B5', 'C#5', 'F#5', 'A5', 'D5', 'E5', 'F#5', 'D5', 'C#5', 'D5', 'E5', 'C#5', 'B4', 'C#5', 'D5', 'B4', 'A4', 'B4', 'C#5', 'A4', 'B4', 'A4', 'A4', 'G#4', 'G#4', 'C#5', 'F#4', 'A4', 'C#5', 'F#5', 'C#5', 'F#5', 'A5', 'F#5', 'F5', 'F#5', 'G#5', 'F#5', 'D5', 'E5', 'F#5', 'C#5', 'B4', 'A4', 'A4', 'G#4', 'G#4', 'C#5', 'F#4', 'C#5', 'F#5', 'A5', 'C#6', 'F#6', 'A6', 'C#7', 'A6', 'D6', 'F6', 'G#6', 'A6', 'B6', 'C#6', 'F#6', 'A6', 'D7', 'E7', 'F#7', 'D7', 'C#7', 'D7', 'E7', 'C#7', 'B6', 'C#7', 'D7', 'B6', 'A6', 'B6', 'C#7', 'A6', 'B6', 'A6', 'A6', 'G#6', 'G#6', 'C#7', 'A6', 'F#6', 'C#7', 'F#7', 'C#6', 'C#6', 'D6', 'C#6', 'B5', 'A#5', 'B5', 'C#6', 'B5', 'A#5', 'C#6', 'B5', 'B5', 'B5', 'C#6', 'B5', 'A5', 'G#5', 'A5', 'B5', 'A5', 'G#5', 'B5', 'A5', 'G#5', 'G#5', 'B5', 'A5', 'G#5', 'F#5', 'F5', 'F#5', 'G#5', 'F#5', 'F5', 'G#5', 'F#5', 'F6', 'G#6', 'C#7', 'C#6', 'D#6', 'F6', 'G#6', 'F#6', 'F#6', 'F6', 'F#6', 'G#6', 'F#

In [4]:
# generate sequences

import numpy as np

input_x = []
output_y = []
seq_len = 5

for i in range(len(note_sequence_numeric) - seq_len):
  input_x.append(note_sequence_numeric[i:i+seq_len])
  output_y.append(note_sequence_numeric[i+seq_len])

input_x = np.array(input_x)
output_y = np.array(output_y)

In [5]:
input_x.shape

(2074, 5)

In [6]:
# define LSTM model architecture

import keras
from keras.models import Sequential
from keras.layers import Input, Embedding, Dense, Dropout, LSTM

model = Sequential()
model.add(Input(shape=(seq_len, )))

# input_dim is vocabulary size which is all possible 128 notes
model.add(Embedding(input_dim=128, output_dim=32))

model.add(LSTM(64, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(64))
model.add(Dropout(0.2))

model.add(Dense(64, activation="relu"))
model.add(Dense(128, activation="softmax"))


In [7]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

In [ ]:
# train model on input sequences to predict next note
model.fit(input_x, output_y, epochs=100)

In [9]:
# prompt to generate music
from pretty_midi.utilities import note_name_to_number
prompt = ['E4', 'G#4', 'A4', 'B4', 'G#4']
prompt_numeric = []
for key in prompt:
  prompt_numeric.append(note_name_to_number(key))

print(prompt_numeric)

[64, 68, 69, 71, 68]


In [10]:
# generate k new notes from given prompt

k = 15
for i in range(k):

  # pass only sequence of expected length
  seed = prompt_numeric[-seq_len:]
  pred_y = model.predict(np.array([seed]))
  y = np.argmax(pred_y)
  prompt_numeric.append(y)

print(prompt_numeric)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 380ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
[64, 68, 69, 71, 68, np.int64(59), np.int64(71), np.int64(65), np.int64(54), np.int64(58), np.int64(61), np.int64(66), np.int64(70), np.int64(73), np.int64(68), np.int64(71), np.int64(66), np.int64(70), np.int64(65), np.int64(68)]


In [11]:
# print new generated sequence with corresponding note names
new_sequence = []

for k in range(len(prompt_numeric)):
  new_sequence.append(note_number_to_name(prompt_numeric[k]))
print(new_sequence)

['E4', 'G#4', 'A4', 'B4', 'G#4', 'B3', 'B4', 'F4', 'F#3', 'A#3', 'C#4', 'F#4', 'A#4', 'C#5', 'G#4', 'B4', 'F#4', 'A#4', 'F4', 'G#4']


In [12]:
# generate MIDI from new note sequence

new_melody = pm.PrettyMIDI()
new_instrument = pm.Instrument(program=49)

end_time = 0
duration = 0.5

for k in range(len(prompt_numeric)):
  note_number = prompt_numeric[k]
  start_time = end_time
  end_time = start_time + duration

  new_note = pm.Note(velocity=100, pitch=note_number, start=start_time, end=end_time)
  new_instrument.notes.append(new_note)

new_melody.instruments.append(new_instrument)
new_melody.write("output.mid")